# Task 13: Neural Networks & Computer Vision

## Introduction to Neural Networks

Neural Networks are computing systems inspired by biological neural networks. They consist of layers of interconnected nodes (neurons) that learn to recognize patterns in data.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 2. Implement Neural Network from Scratch

In [ ]:
class NeuralNetworkFromScratch:
    """
    Simple Neural Network implementation from scratch
    """
    
    def __init__(self, layers, learning_rate=0.01):
        self.layers = layers
        self.learning_rate = learning_rate
        self.weights = []
        self.biases = []
        self._initialize_parameters()
    
    def _initialize_parameters(self):
        np.random.seed(42)
        for i in range(len(self.layers) - 1):
            w = np.random.randn(self.layers[i], self.layers[i+1]) * 0.1
            b = np.zeros((1, self.layers[i+1]))
            self.weights.append(w)
            self.biases.append(b)
    
    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def _sigmoid_derivative(self, z):
        return z * (1 - z)
    
    def _relu(self, z):
        return np.maximum(0, z)
    
    def _relu_derivative(self, z):
        return (z > 0).astype(float)
    
    def _forward(self, X):
        self.activations = [X]
        self.z_values = []
        
        for i in range(len(self.weights)):
            z = np.dot(self.activations[-1], self.weights[i]) + self.biases[i]
            self.z_values.append(z)
            
            if i < len(self.weights) - 1:
                a = self._relu(z)
            else:
                a = self._sigmoid(z)
            self.activations.append(a)
        
        return self.activations[-1]
    
    def _backward(self, X, y):
        m = X.shape[0]
        deltas = []
        
        output_error = y - self.activations[-1]
        output_delta = output_error * self._sigmoid_derivative(self.activations[-1])
        deltas.insert(0, output_delta)
        
        for i in range(len(self.weights) - 2, -1, -1):
            error = deltas[0].dot(self.weights[i+1].T)
            delta = error * self._relu_derivative(self.activations[i+1])
            deltas.insert(0, delta)
        
        for i in range(len(self.weights)):
            self.weights[i] += self.activations[i].T.dot(deltas[i]) / m * self.learning_rate
            self.biases[i] += np.sum(deltas[i], axis=0, keepdims=True) / m * self.learning_rate
    
    def fit(self, X, y, epochs=1000):
        self.loss_history = []
        
        for _ in range(epochs):
            output = self._forward(X)
            self._backward(X, y.reshape(-1, 1))
            
            if _ % 100 == 0:
                loss = np.mean((y - output.flatten()) ** 2)
                self.loss_history.append(loss)
        
        return self
    
    def predict(self, X):
        output = self._forward(X)
        return (output.flatten() > 0.5).astype(int)

## 3. Train Neural Network

In [ ]:
# Generate sample data
X, y = make_classification(n_samples=200, n_features=4, n_classes=2, random_state=42)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train neural network
nn = NeuralNetworkFromScratch(layers=[4, 8, 4, 1], learning_rate=0.1)
nn.fit(X_train_scaled, y_train, epochs=500)

# Predictions
y_pred = nn.predict(X_test_scaled)

print("=== Neural Network (From Scratch) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nLoss History (last 5): {nn.loss_history[-5:]}")

## 4. Neural Network with Scikit-Learn (MLPClassifier)

In [ ]:
from sklearn.neural_network import MLPClassifier

# Create and train MLP
mlp = MLPClassifier(hidden_layer_sizes=(8, 4), activation='relu', 
                    solver='adam', max_iter=500, random_state=42)
mlp.fit(X_train_scaled, y_train)

# Predictions
y_pred_mlp = mlp.predict(X_test_scaled)

print("=== Neural Network (Scikit-Learn MLP) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mlp):.4f}")
print(f"\nLoss curve (last 5): {mlp.loss_curve_[-5:]}")

## 5. Computer Vision with TensorFlow/Keras

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist, cifar10
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")

## 6. MNIST Digit Classification

In [ ]:
# Load MNIST data
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Preprocess
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Number of classes: {10}")

## 7. Build CNN Model

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 8. Train CNN Model

In [ ]:
history = model.fit(X_train, y_train_cat, epochs=5, batch_size=64, 
                    validation_split=0.1, verbose=1)

## 9. Evaluate Model

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\n=== CNN Model Evaluation ===")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

## 10. Visualize Predictions

In [ ]:
# Make predictions
predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=1)

# Visualize some predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'Pred: {predicted_labels[i]}')
    ax.axis('off')
plt.suptitle('MNIST Digit Classification Predictions')
plt.tight_layout()
plt.show()

## 11. Plot Training History

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. CIFAR-10 Image Classification

In [ ]:
# Load CIFAR-10 data
(X_train_cifar, y_train_cifar), (X_test_cifar, y_test_cifar) = cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Preprocess
X_train_cifar = X_train_cifar.astype('float32') / 255.0
X_test_cifar = X_test_cifar.astype('float32') / 255.0
y_train_cifar_cat = to_categorical(y_train_cifar, 10)
y_test_cifar_cat = to_categorical(y_test_cifar, 10)

print(f"CIFAR-10 Training data shape: {X_train_cifar.shape}")
print(f"Classes: {class_names}")

## 13. Build CIFAR-10 Model

In [ ]:
cifar_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

cifar_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

cifar_model.summary()

## 14. Train CIFAR-10 Model

In [ ]:
cifar_history = cifar_model.fit(X_train_cifar, y_train_cifar_cat, 
                                 epochs=5, batch_size=64, 
                                 validation_split=0.1, verbose=1)

## 15. Visualize CIFAR-10 Predictions

In [ ]:
# Make predictions
cifar_predictions = cifar_model.predict(X_test_cifar)
cifar_pred_labels = np.argmax(cifar_predictions, axis=1)

# Visualize
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test_cifar[i])
    ax.set_title(f'Pred: {class_names[cifar_pred_labels[i]]}')
    ax.axis('off')
plt.suptitle('CIFAR-10 Classification Predictions')
plt.tight_layout()
plt.show()

## 16. Summary

### Key Concepts Covered:

1. **Neural Networks**: Layers of interconnected neurons
2. **Activation Functions**: Sigmoid, ReLU
3. **Forward Propagation**: Input to output
4. **Backpropagation**: Learning through error adjustment
5. **CNN Architecture**: Conv2D, MaxPooling, Flatten
6. **Computer Vision**: Image classification with CNNs

### Models Built:
- Neural Network from Scratch
- MLP Classifier (Scikit-Learn)
- CNN for MNIST (Digit Recognition)
- CNN for CIFAR-10 (Image Classification)